In [ ]:
import time, os, sys
START = time.time()

# ---- the only lines you normally edit ----
MODEL = "qwen2.5-7b"          # llama-3.1-8b | qwen2.5-7b | mistral-7b | phi-3.5-mini
LIMIT = 3                     # smoke-test items
ALLOW_UNVERIFIED = True       # set False once the benchmark labels are human-verified
VLLM_SPEC = "vllm"            # if the vLLM install fails once, change to "vllm==0.9.2"
PREV_DIR = "/kaggle/input/prev-results"   # earlier results_*.zip, uploaded as a dataset named prev-results

# Judge (Google Gemini). Values from Google AI Studio for the model you pick.
JUDGE_PROVIDER = "gemini"     # Gemini is the only judge in this project
JUDGE_MODEL  = "gemini-3.1-flash-lite"             # FILL IN: model id exactly as listed (Cell 7b prints the ids your key can use)
JUDGE_FAMILY = "google"       # must differ from llama/qwen/mistral/phi (the harness checks)
JUDGE_MAX_TOKENS = 300        # room for hidden reasoning; some Gemini models cannot turn it off
JUDGE_REASONING_EFFORT = "low"   # Gemini accepts minimal|low|medium|high; None = do not send
PRICE_IN_1M  = 0           # FILL IN: USD per 1M input tokens; 0.0 on the free tier
PRICE_OUT_1M = 0           # FILL IN: USD per 1M output tokens; 0.0 on the free tier
JUDGE_RPM = None              # FILL IN: requests/minute limit shown in AI Studio (use a bit less)
JUDGE_TPM = None              # FILL IN: tokens/minute limit (use a bit less)
JUDGE_RPD = None              # FILL IN: requests/day limit (use a bit less)
# ------------------------------------------

REPO_URL = "https://github.com/Gauravsharma2711/ai-sprints-research.git"
REPO = "/kaggle/working/repo"
PER_MODEL = {   # mirrors config.yaml: (tensor_parallel, vLLM --quantization)
    "llama-3.1-8b": (2, None),
    "qwen2.5-7b":   (1, "awq"),
    "mistral-7b":   (2, None),
    "phi-3.5-mini": (1, None),
}
TP, QUANT = PER_MODEL[MODEL]
JUDGE_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
JUDGE_KEY = "GEMINI_API_KEY"
UNV = "--allow-unverified" if ALLOW_UNVERIFIED else ""
LORA_PATH, LORA_NAME = None, "contract_sft"   # optional LoRA adapter; unused in this project
print("model:", MODEL, "| TP:", TP, "| quant:", QUANT, "| judge:", JUDGE_PROVIDER, JUDGE_MODEL or "(not set)")

In [ ]:
import subprocess, shutil, glob
from kaggle_secrets import UserSecretsClient

def sh(cmd, timeout=None, show=True):
    p = subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True, timeout=timeout)
    if show:
        print(p.stdout[-3000:]); print(p.stderr[-3000:])
    return p.returncode

sec = UserSecretsClient()
for k in ["HF_TOKEN", "GEMINI_API_KEY"]:
    try:
        os.environ[k] = sec.get_secret(k)
        print(k, "loaded")                       # never prints the value
    except Exception:
        print(k, "not attached" + (" - REQUIRED" if k in ("HF_TOKEN", JUDGE_KEY) else " (optional)"))
os.environ["HF_HOME"] = "/tmp/hf"

if not os.path.isdir(REPO):
    assert sh(f"git clone {REPO_URL} {REPO}") == 0, "clone failed: is Internet on?"
else:
    sh(f"cd {REPO} && git checkout -- config.yaml && git pull")   # Cell 3 edits config.yaml
os.chdir(REPO)
sh("git log --oneline -1")
sh(f'"{sys.executable}" -m pip install -q -r requirements.txt')

# Restore earlier results, keeping the layout the harness expects.
raw, traj = f"{REPO}/results/raw", f"{REPO}/results/raw/trajectories"
os.makedirs(traj, exist_ok=True)
if os.path.isdir(PREV_DIR):
    csvs = sorted(glob.glob(f"{PREV_DIR}/**/results.csv", recursive=True), key=os.path.getsize)
    if csvs:
        shutil.copy(csvs[-1], f"{raw}/results.csv")      # the largest = most complete
        print("results.csv from", csvs[-1])
    for f in glob.glob(f"{PREV_DIR}/**/trajectories/*.jsonl", recursive=True):
        dst = f"{traj}/{os.path.basename(f)}"
        if not os.path.exists(dst) or os.path.getsize(f) > os.path.getsize(dst):
            shutil.copy(f, dst)
    print("raw:", sorted(os.listdir(raw)), "| trajectories:", sorted(os.listdir(traj)))
else:
    print("no earlier results attached: starting fresh")

In [ ]:
import yaml
CFG = f"{REPO}/config.yaml"

def _edit(fn):
    c = yaml.safe_load(open(CFG)); fn(c); yaml.safe_dump(c, open(CFG, "w"), sort_keys=False); return c

def apply_settings(c):
    s = next(x for x in c["subjects"] if x["name"] == MODEL)
    assert s["tensor_parallel"] == TP, (s["tensor_parallel"], TP)
    s["confirmed"] = True                      # you have checked the HF id, licence and T4 fit
    c["judge"].update(provider=JUDGE_PROVIDER, model=JUDGE_MODEL or None, family=JUDGE_FAMILY,
                      base_url=JUDGE_BASE_URL, api_key_env=JUDGE_KEY, max_tokens=JUDGE_MAX_TOKENS,
                      reasoning_effort=JUDGE_REASONING_EFFORT,
                      price_per_1m_input_usd=PRICE_IN_1M, price_per_1m_output_usd=PRICE_OUT_1M)
    c["judge"]["rate_limits"].update(requests_per_minute=JUDGE_RPM, tokens_per_minute=JUDGE_TPM,
                                     requests_per_day=JUDGE_RPD)

def set_cap(usd):
    """Call only with an amount YOU approved after reading an --estimate printout."""
    _edit(lambda c: c["budget"].__setitem__("max_spend_usd", float(usd)))
    print("judge spend cap set to $", usd)

def judge_ready():
    assert JUDGE_MODEL and PRICE_IN_1M is not None and PRICE_OUT_1M is not None, \
        "Fill in JUDGE_MODEL and the two prices in Cell 1, then rerun Cells 1 and 3."

cfg = _edit(apply_settings)
HF_MODEL = next(x for x in cfg["subjects"] if x["name"] == MODEL)["hf_id"]
print("will serve:", HF_MODEL, "| judge:", JUDGE_PROVIDER, JUDGE_MODEL or "(not set)",
      "| cap: $", cfg["budget"]["max_spend_usd"])

In [ ]:
import sys, torch
sh("nvidia-smi")
print(sys.version)
assert sys.version_info >= (3, 11), "the harness needs Python 3.11+"
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPUs:", torch.cuda.device_count())

In [ ]:
t0 = time.time()
rc = sh("pip install -q uv && uv venv /tmp/vllm-env --seed && "
        f"uv pip install --python /tmp/vllm-env/bin/python '{VLLM_SPEC}' --torch-backend=cu128",
        timeout=1200)
print("rc:", rc, "| minutes:", round((time.time() - t0) / 60, 1))
sh("/tmp/vllm-env/bin/python -c \"import vllm, torch; print(vllm.__version__, torch.version.cuda, torch.cuda.is_available())\"")

In [ ]:
import json, urllib.request
env = dict(os.environ, VLLM_ATTENTION_BACKEND="XFORMERS", HF_HOME="/tmp/hf")
cmd = ["/tmp/vllm-env/bin/vllm", "serve", HF_MODEL, "--dtype", "half", "--max-model-len", "4096",
       "--gpu-memory-utilization", "0.88", "--tensor-parallel-size", str(TP), "--port", "8000"]
if QUANT:
    cmd += ["--quantization", QUANT]
if LORA_PATH:
    cmd += ["--enable-lora", "--lora-modules", f"{LORA_NAME}={LORA_PATH}", "--max-lora-rank", "64"]
if TP == 1:
    env["CUDA_VISIBLE_DEVICES"] = "0"

server = subprocess.Popen(cmd, env=env, stdout=open("/kaggle/working/vllm_server.log", "w"),
                          stderr=subprocess.STDOUT)
ready = False
for i in range(180):
    if server.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://localhost:8000/v1/models", timeout=3) as r:
            print("READY:", [m["id"] for m in json.load(r)["data"]]); ready = True; break
    except Exception:
        pass
    time.sleep(5)
if not ready:
    print("NOT READY. Log tail:\n", open("/kaggle/working/vllm_server.log").read()[-3000:])

In [ ]:
from openai import OpenAI
# a) local logprobs (free)
local = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
r = local.chat.completions.create(model=LORA_NAME if LORA_PATH else HF_MODEL,
    messages=[{"role": "user", "content": "Answer with one word, Yes or No: Is the sky blue on a clear day?"}],
    max_tokens=1, temperature=0, logprobs=True, top_logprobs=5)
print("logprobs:", [(t.token, round(t.logprob, 3)) for t in r.choices[0].logprobs.content[0].top_logprobs])

In [ ]:
# b) judge: list the model ids your key can use (free), then one tiny test call
judge = OpenAI(base_url=JUDGE_BASE_URL, api_key=os.environ[JUDGE_KEY])
try:
    print("models:", sorted(m.id for m in judge.models.list()))
except Exception as e:
    print("could not list models here; copy the id from AI Studio:", type(e).__name__)
judge_ready()
extra = {"reasoning_effort": JUDGE_REASONING_EFFORT} if JUDGE_REASONING_EFFORT else {}
j = judge.chat.completions.create(model=JUDGE_MODEL, temperature=0, max_tokens=JUDGE_MAX_TOKENS,
    messages=[{"role": "user", "content": "Reply exactly: PROBABILITY_YES: 0.50"}], **extra)
print("judge:", repr(j.choices[0].message.content), "| finish:", j.choices[0].finish_reason, "|", j.usage)
assert "PROBABILITY_YES" in (j.choices[0].message.content or ""), "empty judge reply: raise JUDGE_MAX_TOKENS"

In [ ]:
# c) smoke-test estimate (free): call counts and worst-case judge cost
!"{sys.executable}" -m src.harness --models {MODEL} --limit {LIMIT} --out-dir results/smoke --estimate

In [ ]:
t1 = time.time()
!"{sys.executable}" -u -m src.harness --models {MODEL} --limit {LIMIT} --out-dir results/smoke --approve-spend {UNV}
smoke_seconds = time.time() - t1
print("smoke seconds:", round(smoke_seconds))

In [ ]:
# full transcript of every arm for one cell
!"{sys.executable}" -m src.harness --show-histories --out-dir results/smoke --condition contract_prompt

In [ ]:
if "smoke_seconds" in globals():           # set by Cell 8; not present in a resumed session
    per_item = smoke_seconds / LIMIT      # one item = 3 conditions x 3 samples x 11 calls
    print(f"~{per_item:.0f}s per item -> ~{per_item * 60 / 3600:.1f} h for all 60 items of {MODEL}")
else:
    print("no smoke test in this session: skipping the time estimate")
!"{sys.executable}" -m src.harness --models {MODEL} --skip-judge --estimate

In [ ]:
!"{sys.executable}" -u -m src.harness --models {MODEL} --skip-judge {UNV}

In [ ]:
judge_ready()
!"{sys.executable}" -m src.harness --judge-only --estimate

In [ ]:
!"{sys.executable}" -u -m src.harness --judge-only --approve-spend

In [ ]:
import pandas as pd
server.terminate()
tag = MODEL + ("_lora" if LORA_PATH else "")
shutil.copy("/kaggle/working/vllm_server.log", f"{REPO}/results/raw/vllm_server_{tag}.log")
shutil.make_archive(f"/kaggle/working/results_{tag}", "zip", f"{REPO}/results/raw")

df = pd.read_csv(f"{REPO}/results/raw/results.csv")
print("rows:", len(df), "| by model:", df["subject"].value_counts().to_dict())
print("judge status:", df["status_judge"].value_counts().to_dict())
st = [c for c in df.columns if c.startswith("status_")]
print("share not ok, per arm:\n", (df[st] != "ok").mean().round(3).sort_values(ascending=False))
print("session hours:", round((time.time() - START) / 3600, 2))